In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('Advertising.csv')
print(df.shape)
print(df.head())

(200, 4)
      TV  Radio  Newspaper  Sales
0  230.1   37.8       69.2   22.1
1   44.5   39.3       45.1   10.4
2   17.2   45.9       69.3   12.0
3  151.5   41.3       58.5   16.5
4  180.8   10.8       58.4   17.9


In [3]:
from sklearn.model_selection import train_test_split

X_full = df[['TV', 'Radio', 'Newspaper']]
Y_full = df['Sales']

X_train, X_test, Y_train, Y_test = train_test_split(
    X_full, Y_full, test_size=0.30, random_state=1
)

print(f"Entrenamiento: {X_train.shape[0]} observaciones")
print(f"Prueba: {X_test.shape[0]} observaciones")

Entrenamiento: 140 observaciones
Prueba: 60 observaciones


In [4]:
import statsmodels.api as sm

X_train_const = sm.add_constant(X_train)
modelo_completo = sm.OLS(Y_train, X_train_const).fit()
print(modelo_completo.summary())

                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.899
Model:                            OLS   Adj. R-squared:                  0.897
Method:                 Least Squares   F-statistic:                     405.2
Date:                Wed, 14 Oct 2026   Prob (F-statistic):           1.36e-67
Time:                        13:06:16   Log-Likelihood:                -272.35
No. Observations:                 140   AIC:                             552.7
Df Residuals:                     136   BIC:                             564.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.6614      0.368     12.650      0.0

In [5]:
X_train_final = X_train[['TV', 'Radio']]
X_test_final  = X_test[['TV', 'Radio']]

n_train = len(Y_train)
X = np.column_stack([np.ones(n_train), X_train_final.values])  # [1, TV, Radio]
Y = Y_train.values.reshape(-1, 1)
print("Dimensiones X:", X.shape, "| Dimensiones Y:", Y.shape)

Dimensiones X: (140, 3) | Dimensiones Y: (140, 1)


In [6]:
# X'X
XT_X = np.matmul(np.matrix.transpose(X), X)
print("X'X:")
print(XT_X)

X'X:
[[1.40000000e+02 1.99806000e+04 3.24870000e+03]
 [1.99806000e+04 3.90127308e+06 4.70424050e+05]
 [3.24870000e+03 4.70424050e+05 1.07757590e+05]]


In [7]:
XT_X_inv = np.linalg.inv(XT_X)
print("(X'X)^-1:")
print(XT_X_inv)

(X'X)^-1:
[[ 4.19082008e-02 -1.31516536e-04 -6.89312286e-04]
 [-1.31516536e-04  9.53968372e-07 -1.99632300e-07]
 [-6.89312286e-04 -1.99632300e-07  3.09331404e-05]]


In [8]:
XT_Y = np.matmul(np.matrix.transpose(X), Y)
print("X'Y:")
print(XT_Y)

X'Y:
[[  2078.7 ]
 [355094.55]
 [ 51899.21]]


In [9]:
betas = np.matmul(XT_X_inv, XT_Y)
print("Betas [intercepto, TV, Radio]:")
print(betas)

Betas [intercepto, TV, Radio]:
[[4.63900893]
 [0.05500479]
 [0.10164376]]


In [10]:
Y_pred = np.matmul(X, betas)
Resid = Y - Y_pred

RSS = np.matmul(np.matrix.transpose(Resid), Resid)
TSS = np.matmul(np.matrix.transpose(Y), Y) - len(Y) * (Y.mean() ** 2)

R_cuad = float(1 - RSS / TSS)
print("RSS:", RSS)
print("TSS:", TSS)
print("R² entrenamiento:", R_cuad)

s_cuad = RSS / (len(Y) - X.shape[1])
import math
s = math.sqrt(float(s_cuad))
print("s²:", s_cuad)
print("s :", s)

RSS: [[401.35084462]]
TSS: [[3987.34935714]]
R² entrenamiento: 0.8993439479032658
s²: [[2.92956821]]
s : 1.7115981446906272


## R2 de Prueba

In [13]:
n_test = len(Y_test)
X_test_mat = np.column_stack([np.ones(n_test), X_test_final.values])
Y_test_mat = Y_test.values.reshape(-1, 1)

Y_pred_test = np.matmul(X_test_mat, betas)
RSS_test = np.matmul(np.matrix.transpose(Y_test_mat - Y_pred_test),
                     (Y_test_mat - Y_pred_test))
TSS_test = np.matmul(np.matrix.transpose(Y_test_mat), Y_test_mat) \
           - n_test * (Y_test_mat.mean() ** 2)
R2_test = float(1 - RSS_test / TSS_test)
print("R² prueba:", R2_test)

R² prueba: 0.9073562242286405


## Intervalo de Confiaza

In [14]:
import scipy.stats

f = np.array([[1], [100], [50]])

grados_libertad = len(Y) - X.shape[1]      # 140 - 3 = 137
Confianza = 0.90
q = (1 - Confianza) / 2
t_critico = abs(scipy.stats.t.ppf(q, df=grados_libertad))
print("t crítico (90%, gl=137):", t_critico)

Margen_error = t_critico * (s * (float(np.matmul(np.matmul(np.matrix.transpose(f), XT_X_inv), f)) ** 0.5))
Pron_puntual = float(np.matmul(np.matrix.transpose(f), betas))

Lim_inferior = Pron_puntual - Margen_error
Lim_superior = Pron_puntual + Margen_error

print("Pronóstico puntual:", Pron_puntual)
print(f"Intervalo de confianza 90%: [{Lim_inferior:.4f}, {Lim_superior:.4f}]")

t crítico (90%, gl=137): 1.6560520804924401
Pronóstico puntual: 15.221675664877207
Intervalo de confianza 90%: [14.7182, 15.7251]


## Sesgo, Curtosis y Jarque-Bera

In [15]:
skew_scipy = float(scipy.stats.skew(Resid, bias=True))
kurtosis_scipy = float(scipy.stats.kurtosis(Resid, fisher=False))
print(f"Sesgo (scipy):    {skew_scipy}")
print(f"Curtosis (scipy): {kurtosis_scipy}")

Sesgo (scipy):    -0.5124182186793586
Curtosis (scipy): 4.782590640126667


In [16]:
def skewness_manual(x):
    x = np.array(x).flatten()
    n = len(x)
    mean_x = np.mean(x)
    m3 = np.sum((x - mean_x) ** 3) / n
    s3 = (np.sum((x - mean_x) ** 2) / n) ** 1.5
    return m3 / s3

def kurtosis_manual(x):
    x = np.array(x).flatten()
    n = len(x)
    mean_x = np.mean(x)
    m4 = np.sum((x - mean_x) ** 4) / n
    s4 = (np.sum((x - mean_x) ** 2) / n) ** 2
    return m4 / s4

skew_manual = skewness_manual(Resid)
kurt_manual = kurtosis_manual(Resid)

print(f"Sesgo manual:    {skew_manual}")
print(f"Sesgo scipy:     {skew_scipy}")
print(f"Diferencia:      {abs(skew_manual - skew_scipy)}")
print()
print(f"Curtosis manual: {kurt_manual}")
print(f"Curtosis scipy:  {kurtosis_scipy}")
print(f"Diferencia:      {abs(kurt_manual - kurtosis_scipy)}")

Sesgo manual:    -0.5124182186793586
Sesgo scipy:     -0.5124182186793586
Diferencia:      0.0

Curtosis manual: 4.782590640126667
Curtosis scipy:  4.782590640126667
Diferencia:      0.0


## Jarque-Bera manual vs scipy

In [17]:
JB_manual = (len(Y) / 6) * (skew_manual ** 2 + ((kurt_manual - 3) ** 2) / 4)
JB_scipy, p_JB_scipy = scipy.stats.jarque_bera(Resid)

chi2_critico = scipy.stats.chi2.ppf(0.90, df=2)

print(f"JB manual:  {JB_manual}")
print(f"JB scipy:   {JB_scipy}, p = {p_JB_scipy}")
print(f"Valor crítico chi²(2) al 90%: {chi2_critico}")

JB manual:  24.662861496030963
JB scipy:   24.662861496030963, p = 4.410904674141494e-06
Valor crítico chi²(2) al 90%: 4.605170185988092


## Durbin-Watson

In [18]:
from statsmodels.stats.stattools import durbin_watson
dw_statsmodels = durbin_watson(Resid.flatten())
print(f"DW statsmodels: {dw_statsmodels}")

DW statsmodels: 2.0375161916374984


In [19]:
def durbin_watson_manual(residuos):
    residuos = np.array(residuos).flatten()
    diff = np.diff(residuos)
    return np.sum(diff ** 2) / np.sum(residuos ** 2)

dw_manual = durbin_watson_manual(Resid)
print(f"DW manual:      {dw_manual}")
print(f"DW statsmodels: {dw_statsmodels}")
print(f"Diferencia:     {abs(dw_manual - dw_statsmodels)}")

DW manual:      2.0375161916374984
DW statsmodels: 2.0375161916374984
Diferencia:     0.0


## Prueba de White

In [20]:
from statsmodels.stats.diagnostic import het_white

X_train_sm = sm.add_constant(X_train_final)
modelo_sm = sm.OLS(Y_train, X_train_sm).fit()

white_test = het_white(modelo_sm.resid, modelo_sm.model.exog)
print("White (statsmodels): LM =", white_test[0], "| p =", white_test[1])

White (statsmodels): LM = 11.353378151633706 | p = 0.04480664163871973


In [21]:
def white_test_manual(X_vars, residuos):
    residuos = np.array(residuos).flatten()
    n = len(residuos)
    resid2 = residuos ** 2
    X_arr = np.array(X_vars)
    p = X_arr.shape[1]
    
    cols = [np.ones(n)]
    for i in range(p):
        cols.append(X_arr[:, i])
    for i in range(p):
        cols.append(X_arr[:, i] ** 2)
    for i in range(p):
        for j in range(i + 1, p):
            cols.append(X_arr[:, i] * X_arr[:, j])
    
    X_aux = np.column_stack(cols)
    
    XtX = np.matmul(X_aux.T, X_aux)
    XtX_inv = np.linalg.inv(XtX)
    Xty = np.matmul(X_aux.T, resid2)
    beta_aux = np.matmul(XtX_inv, Xty)
    y_pred_aux = np.matmul(X_aux, beta_aux)
    resid_aux = resid2 - y_pred_aux
    
    RSS_aux = np.sum(resid_aux ** 2)
    TSS_aux = np.sum((resid2 - resid2.mean()) ** 2)
    R2_aux = 1 - RSS_aux / TSS_aux
    
    LM = n * R2_aux
    df = X_aux.shape[1] - 1
    p_value = 1 - scipy.stats.chi2.cdf(LM, df)
    return LM, p_value, R2_aux

LM_manual, p_manual, R2_aux_manual = white_test_manual(X_train_final.values, Resid)

print(f"White manual:      LM = {LM_manual:.4f}, p = {p_manual:.4f}, R²_aux = {R2_aux_manual:.4f}")
print(f"White statsmodels: LM = {white_test[0]:.4f}, p = {white_test[1]:.4f}")

p_vars = X_train_final.shape[1]
gl_white = p_vars * (p_vars + 1) // 2   # 3
chi2_white = scipy.stats.chi2.ppf(0.90, df=gl_white)
print(f"Valor crítico chi²({gl_white}) al 90%: {chi2_white}")

White manual:      LM = 11.3534, p = 0.0448, R²_aux = 0.0811
White statsmodels: LM = 11.3534, p = 0.0448
Valor crítico chi²(3) al 90%: 6.251388631170325


## VIF

In [22]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = sm.add_constant(X_train_final)
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i)
                   for i in range(X_vif.shape[1])]
print(vif_data)

  Variable       VIF
0    const  5.867148
1       TV  1.001352
2    Radio  1.001352
